In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt

import rclpy

from bng_controller import TorqueSpeedController

from bng_simulator.utils.logger_utils import load_run_data

In [ ]:
# Pick your vehicle name as it appears in the scenario YAML (e.g., 'ego')
VEHICLE = 'EGO'

if not rclpy.ok():
    rclpy.init()
ctl = TorqueSpeedController(vehicle_name=VEHICLE, spin_in_thread=True)

In [ ]:
# Collect some data o which we will be doing analysis
# Set a velocity profile and drive around
def velocity_profile(t):
    if t < 15.0:
        return 4.0  # m/s
    elif t < 30.0:
        return 7.0  # m/s
    elif t < 45.0:
        return 10.0  # m/s
    else:
        return 0.0  # m/s

def velocity_profile(t):
    return 5.0  # m/s constant

def steering_profile(t):
    # Min and Max [-0.8, 0.8] and period of 20s starting at 0
    Tp = 20.0
    max_in  = 0.8
    min_in  = -0.8
    return 0.5 * (max_in - min_in) * np.sin(2.0 * np.pi * t / Tp) + 0.5 * (max_in + min_in)

HZ = 50.0
DT = 1.0 / HZ
T_END = 60.0 # Whatever you choose
elapsed = 0
while rclpy.ok():
    # Does not care about drift in time
    if elapsed > T_END:
        break

    target_speed = velocity_profile(elapsed)
    steering_input = steering_profile(elapsed)
    ctl.send_command(vehicle_speed=target_speed, steering=None)

    elapsed += DT
    # Busy wait to maintain timing
    time.sleep(DT)

In [ ]:
# Vehicle parameters
# a  lf, b  lr in meters
a = 1.458
b = 1.136
# Does these values make sense kowing it's a rear wheel drive?

In [ ]:
# Load data
# Pick a run number (1 -> run_001, 2 -> run_002, ...)
RUN_NUMBER = 5  # <-- change me

data = load_run_data(run_number=RUN_NUMBER)

print('Loaded topics/messages:', len(data))
print('First 30 keys:')
for k in list(data.keys())[:30]:
    print('  ', k)

In [ ]:
for k, v in data['/EGO/gtstate'].items():
    print(f'Topic: {k}, Number of messages: {len(v)}')

In [ ]:
# First convert steering input to roadwheel angle and vice versa
gt_data  = data['/EGO/gtstate']
delta = (np.array(gt_data['wheelFR_angleAtan2']) + np.array(gt_data['wheelFL_angleAtan2'])) / 2.0  # in radians
steering_input = np.array(gt_data['steeringInput'])

# Simple linear estimation
slope, offset = np.polyfit(delta, steering_input, 1)
print(f'Estimated steeringToInput: slope={slope}, offset={offset}')
print(f"steering_input  = {slope:.6f} * delta + {offset:.6f}")
print(f"delta           = {(1/slope):.6f} * steering_input - {offset/slope:.6f}") 

In [ ]:
# Velocity profile to test out kinematic relationships
# from pure rolling acceleration Vy = - r * b and r = (Vx / (a + b)) * tan (delta)
# And vy - ar =  vx tan(delta)

vy = np.array(gt_data['vel_y']) # m/s
vx = np.array(gt_data['vel_x']) # m/s
r  = np.array(gt_data['angVel_z']) # rad/s
r_est = (vx / (a + b)) * np.tan(delta)
vy_est =  b * r
# front wheel relationship
vfy_wheel_est = vx * np.tan(delta)
vfy_wheel = vy + a * r


# Plot results
plt.figure(figsize=(12, 9))
plt.subplot(4, 1, 1)
plt.title('Yaw rate estimation')
plt.plot(r, label='Measured r')
plt.plot(r_est, label='Estimated r', linestyle='--')
plt.ylabel('Yaw rate r (rad/s)')
plt.legend()

plt.subplot(4, 1, 2)
plt.title('Lateral velocity estimation')
plt.plot(vy, label='Measured Vy')
plt.plot(vy_est, label='Estimated Vy', linestyle='--')
plt.ylabel('Lateral velocity Vy (m/s)')
plt.legend()

# Just longitudinal velocity profile
plt.subplot(4, 1, 3)
plt.title('Longitudinal velocity profile')
plt.plot(vx, label='Measured Vx')
plt.ylabel('Longitudinal velocity Vx (m/s)')
plt.legend()


# plt.subplot(4, 1, 4)
# plt.title('Front wheel lateral velocity estimation')
# plt.plot(vfy_wheel, label='Measured Vfy_wheel')
# plt.plot(vfy_wheel_est, label='Estimated Vfy_wheel', linestyle='--')
# plt.ylabel('Front wheel lateral velocity Vfy_wheel (m/s)')
# plt.legend()
# plt.xlabel('Time step')
# plt.tight_layout()
# plt.show()

plt.subplot(4, 1, 4)
plt.title('Delta')
plt.plot(delta, label='Measured delta')
plt.legend()
plt.xlabel('Time step')
plt.tight_layout()
plt.show()

# Observation: the estimated yaw rate does not match well the measured one at higher steering angles.
# Find thresholds where the linear model breaks down.
# Possible reasons: tire slip, lateral load transfer, other dynamic effects not captured by the simple


In [ ]:
# Pure pursuit model.
# We need to generate a reference path to follow

# Load data
# Pick a run number (1 -> run_001, 2 -> run_002, ...)
RUN_NUMBER = 6  # <-- change me

data_map = load_run_data(run_number=RUN_NUMBER)

In [ ]:
# Let's visualize x and y positions
gt_data_map  = data_map['/EGO/gtstate']
t = np.array(gt_data_map['time'])  # ignore first 5 seconds
t = t - t[0]  # reset time
t_mask = np.logical_and(t >= 57, True)
x = np.array(gt_data_map['pos_x']) # m
y = np.array(gt_data_map['pos_y']) # m
plt.figure(figsize=(8, 8))
plt.plot(x[t_mask], y[t_mask], label='Vehicle path')
plt.title('Vehicle path in XY plane')
plt.xlabel('X position (m)')
plt.ylabel('Y position (m)')
plt.axis('equal')
plt.legend()
plt.show()

In [ ]:
# Let's see x and y as a function of time
time = np.array(gt_data_map['time'])  # Assuming there's a time field
plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(time, x, label='X position')
plt.title('X position over time')
plt.xlabel('Time (s)')
plt.ylabel('X position (m)')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(time, y, label='Y position', color='orange')
plt.title('Y position over time')
plt.xlabel('Time (s)')
plt.ylabel('Y position (m)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Reverse the mapping to get the reference path
x_interest = np.array(x[t_mask])[::-1]
y_interest = np.array(y[t_mask])[::-1]

# Let's convert quaternion into yaw angle
import scipy.spatial.transform
quat_x = np.array(gt_data_map['quat_x'])[t_mask][::-1]
quat_y = np.array(gt_data_map['quat_y'])[t_mask][::-1]
quat_z = np.array(gt_data_map['quat_z'])[t_mask][::-1]
quat_w = np.array(gt_data_map['quat_w'])[t_mask][::-1]
yaw_angle = np.atan2(
    2.0 * (quat_w * quat_z + quat_x * quat_y),
    1.0 - 2.0 * (quat_y**2 + quat_z**2)
)

In [ ]:
# Now I need to define a proper frenet-ish corresponding to this.
# I dont need the full (s,e,dphi)
# The only thing I need is a ref parametrc (s, x_ref(s), y_ref(s)) so that
# I can compute the target point at a lookahead distance Ld, or my current ref given s position

def get_closest_point(x_ref, y_ref, x_pos, y_pos):
    """ Returns the index of the closest point in the reference path to the current position. """
    distances = np.sqrt((x_ref - x_pos)**2 + (y_ref - y_pos)**2)
    return np.argmin(distances)


## Path preprocessing (smooth + resample in arc-length)

Pure pursuit is geometric: it works best with a path sampled uniformly in distance (arc-length), not uniformly in time.

**Goal:** turn recorded `(x, y)` points into a clean reference `(s, x(s), y(s), yaw(s))`.

Steps:
1) Remove duplicate/near-duplicate points
2) Compute cumulative arc-length `s`
3) Fit a *smoothed* spline `x(s), y(s)`
4) Resample at uniform spacing `ds` (e.g. 0.2–1.0 m)

Tune `ds` (path granularity) and `SPLINE_S` (smoothing strength) until the XY plot looks clean but corners are still preserved.

In [ ]:
# Path smoothing/resampling utilities (arc-length parameterized)
import numpy as np
import matplotlib.pyplot as plt

try:
    from scipy.interpolate import splprep, splev
except Exception as e:
    raise ImportError(
        "This cell requires scipy (scipy.interpolate). Install with: pip install scipy"
    ) from e


def _dedupe_by_distance(x: np.ndarray, y: np.ndarray, min_dist: float = 1e-3):
    """Drop near-duplicate consecutive points to avoid spline degeneracy."""
    x = np.asarray(x).astype(float)
    y = np.asarray(y).astype(float)
    if len(x) < 2:
        return x, y
    dx = np.diff(x)
    dy = np.diff(y)
    ds = np.hypot(dx, dy)
    keep = np.ones(len(x), dtype=bool)
    keep[1:] = ds > min_dist
    return x[keep], y[keep]


def _arc_length(x: np.ndarray, y: np.ndarray):
    """Cumulative arc-length s for polyline points."""
    dx = np.diff(x)
    dy = np.diff(y)
    ds = np.hypot(dx, dy)
    s = np.concatenate([[0.0], np.cumsum(ds)])
    return s


def smooth_resample_path_spline(
    x: np.ndarray,
    y: np.ndarray,
    ds: float = 0.5,
    spline_s: float = 5.0,
    k: int = 3,
    min_dist: float = 1e-3,
    per: bool = False,
    return_debug: bool = False,
):
    """
    Smooth a 2D path using a parametric spline x(s), y(s) and resample at uniform arc-length spacing.

    Parameters
    ----------
    ds : float
        Resampling distance [m] between successive reference points.
    spline_s : float
        Smoothing factor passed to scipy.interpolate.splprep.
        Larger => smoother (more deviation allowed from raw points).
    k : int
        Spline degree (3=cubic). Will be reduced automatically if too few points.
    per : bool
        If True, enforces periodic spline (closed loop).
    """
    x0, y0 = _dedupe_by_distance(x, y, min_dist=min_dist)
    if len(x0) < 4:
        raise ValueError(f"Not enough points after dedupe: {len(x0)}")

    s0 = _arc_length(x0, y0)
    total_length = float(s0[-1])
    if total_length <= 0.0:
        raise ValueError("Total path length is zero; check input points.")

    k_eff = int(min(k, len(x0) - 1))
    if k_eff < 1:
        raise ValueError("Not enough points for spline fitting.")

    # Fit parametric spline with u=s0 so the parameter is true arc-length (meters).
    tck, u = splprep([x0, y0], u=s0, s=spline_s, k=k_eff, per=per)

    s_grid = np.arange(0.0, total_length, ds)
    if s_grid[-1] < total_length:
        s_grid = np.concatenate([s_grid, [total_length]])

    x_s, y_s = splev(s_grid, tck, der=0)
    dx_s, dy_s = splev(s_grid, tck, der=1)
    ddx_s, ddy_s = splev(s_grid, tck, der=2)

    yaw_s = np.arctan2(dy_s, dx_s)
    denom = (np.asarray(dx_s) ** 2 + np.asarray(dy_s) ** 2) ** 1.5
    denom = np.maximum(denom, 1e-9)
    curvature_s = (np.asarray(dx_s) * np.asarray(ddy_s) - np.asarray(dy_s) * np.asarray(ddx_s)) / denom

    result = {
        "s": s_grid,
        "x": np.asarray(x_s),
        "y": np.asarray(y_s),
        "yaw": np.unwrap(np.asarray(yaw_s)),
        "curvature": np.asarray(curvature_s),
        "length": total_length,
    }
    if return_debug:
        result["raw"] = {"x": x0, "y": y0, "s": s0}
        result["tck"] = tck
    return result


# --- Use it on your current extracted segment ---
# Tune these two knobs:
DS = 1.0         # [m] target spacing for the reference path
SPLINE_S = 50.0   # smoothing strength (bigger => smoother)

path = smooth_resample_path_spline(
    x_interest,
    y_interest,
    ds=DS,
    spline_s=SPLINE_S,
    per=False,
    return_debug=True,
)

print(f"Raw points: {len(path['raw']['x'])}")
print(f"Resampled points: {len(path['x'])}")
print(f"Path length: {path['length']:.2f} m")

# Plot raw vs smoothed/resampled
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(path['raw']['x'], path['raw']['y'], alpha=0.4, label='raw')
plt.plot(path['x'], path['y'], '-', linewidth=2, label='smooth+resampled')
plt.axis('equal')
plt.grid(True)
plt.title('XY path')
plt.legend()

plt.subplot(2, 2, 2)
plt.plot(path['s'], path['yaw'])
plt.grid(True)
plt.title('Yaw vs s')
plt.xlabel('s [m]')
plt.ylabel('yaw [rad]')

plt.subplot(2, 2, 4)
plt.plot(path['s'], path['curvature'])
plt.grid(True)
plt.title('Curvature vs s')
plt.xlabel('s [m]')
plt.ylabel('kappa [1/m]')

plt.tight_layout()
plt.show()

In [ ]:
print("Initial Points (x,y, yaw):", path['x'][0], path['y'][0], path['yaw'][0])

In [ ]:
# Let's export it as a csv
import pandas as pd
df = pd.DataFrame({
    's': path['s'],
    'x': path['x'],
    'y': path['y'],
    'yaw': path['yaw'],
    'curvature': path['curvature'],
})
df.to_csv('../../../../labs/week4/ref.csv', index=False)

## Pure Pursuit (modular implementation)

We assume you have a **reference path** sampled by arc-length: arrays `s_ref`, `x_ref`, `y_ref` (like the `path[...]` produced above).

At each control step, given the vehicle pose `(x, y, yaw)` and a lookahead distance `Ld`:
- Project `(x, y)` onto the polyline to estimate `s_closest` (optionally using `last_s` to avoid jumping to a different lap).
- Set `s_target = s_closest + Ld` and interpolate the target point `(x_t, y_t)`.
- Compute heading error to target: `alpha = wrap(atan2(y_t - y, x_t - x) - yaw)`.
- Pure pursuit curvature command: $\kappa = \frac{2\sin(\alpha)}{L_d}$.
- Convert to steering angle (bicycle model): $\delta = \arctan(L\,\kappa)$, where `L` is wheelbase.

In [ ]:
import numpy as np


def wrap_angle_pi(angle_rad: float) -> float:
    """Wrap angle to [-pi, pi)."""
    return (angle_rad + np.pi) % (2.0 * np.pi) - np.pi


# def interp1d_linear(x: np.ndarray, xp: np.ndarray, fp: np.ndarray) -> np.ndarray:
#     """Numpy linear interpolation wrapper with float arrays."""
#     return np.interp(np.asarray(x, dtype=float), np.asarray(xp, dtype=float), np.asarray(fp, dtype=float))


def target_point_from_s(s_ref: np.ndarray, x_ref: np.ndarray, y_ref: np.ndarray, yaw_ref: np.ndarray | None, s_query: float):
    """Interpolate reference at arc-length s_query."""
    s_query = float(np.clip(s_query, s_ref[0], s_ref[-1]))
    x_t = float(np.interp(s_query, s_ref, x_ref))
    y_t = float(np.interp(s_query, s_ref, y_ref))
    if yaw_ref is None:
        return x_t, y_t, None
    # yaw_ref should be unwrapped already for meaningful interpolation
    yaw_t = float(np.interp(s_query, s_ref, yaw_ref))
    return x_t, y_t, yaw_t


def project_point_to_polyline(
    x_ref: np.ndarray,
    y_ref: np.ndarray,
    s_ref: np.ndarray,
    x: float,
    y: float,
    i0: int = 0,
    i1: int | None = None,
):
    """
    Project a point (x,y) onto a polyline defined by consecutive segments of (x_ref,y_ref).
    Returns (s_proj, x_proj, y_proj, seg_index, dist).
    """
    x_ref = np.asarray(x_ref, dtype=float)
    y_ref = np.asarray(y_ref, dtype=float)
    s_ref = np.asarray(s_ref, dtype=float)
    n = len(x_ref)
    if i1 is None:
        i1 = n - 1
    i0 = int(np.clip(i0, 0, n - 2))
    i1 = int(np.clip(i1, i0 + 1, n - 1))

    best = {"dist": float("inf"), "s": s_ref[i0], "x": x_ref[i0], "y": y_ref[i0], "i": i0}
    px = float(x)
    py = float(y)
    eps = 1e-12

    for i in range(i0, i1):
        ax, ay = x_ref[i], y_ref[i]
        bx, by = x_ref[i + 1], y_ref[i + 1]
        vx, vy = (bx - ax), (by - ay)
        vv = vx * vx + vy * vy
        if vv < eps:
            continue
        t = ((px - ax) * vx + (py - ay) * vy) / vv
        t = float(np.clip(t, 0.0, 1.0))
        proj_x = ax + t * vx
        proj_y = ay + t * vy
        dist = float(np.hypot(px - proj_x, py - proj_y))
        if dist < best["dist"]:
            s_proj = float(s_ref[i] + t * (s_ref[i + 1] - s_ref[i]))
            best = {"dist": dist, "s": s_proj, "x": float(proj_x), "y": float(proj_y), "i": i}

    return best["s"], best["x"], best["y"], best["i"], best["dist"]


def _index_range_from_s_window(s_ref: np.ndarray, s_center: float, s_back: float, s_fwd: float):
    """Compute [i0,i1] segment index window from s window."""
    s0 = float(np.clip(s_center - s_back, s_ref[0], s_ref[-1]))
    s1 = float(np.clip(s_center + s_fwd, s_ref[0], s_ref[-1]))
    # Convert s bounds to point indices; we’ll project onto segments, so end is -1
    i0 = int(np.searchsorted(s_ref, s0, side="left")) - 1
    i1 = int(np.searchsorted(s_ref, s1, side="right"))
    i0 = int(np.clip(i0, 0, len(s_ref) - 2))
    i1 = int(np.clip(i1, i0 + 1, len(s_ref) - 1))
    return i0, i1


def pure_pursuit_step(
    s_ref: np.ndarray,
    x_ref: np.ndarray,
    y_ref: np.ndarray,
    yaw_ref: np.ndarray | None,
    x: float,
    y: float,
    yaw: float,
    Ld: float,
    wheelbase_L: float,
    last_s: float | None = None,
    s_back: float = 5.0,
    s_fwd: float = 30.0,
):
    """
    One pure pursuit step.

    - If last_s is provided, projection searches only within [last_s - s_back, last_s + s_fwd] to avoid jumps.
    - Returns dict with s_closest, s_target, target point, curvature and steering angle.
    """
    s_ref = np.asarray(s_ref, dtype=float)
    x_ref = np.asarray(x_ref, dtype=float)
    y_ref = np.asarray(y_ref, dtype=float)
    if yaw_ref is not None:
        yaw_ref = np.asarray(yaw_ref, dtype=float)

    if Ld <= 1e-6:
        raise ValueError("Ld must be positive")
    if wheelbase_L <= 1e-6:
        raise ValueError("wheelbase_L must be positive")

    if last_s is None:
        i0, i1 = 0, len(s_ref) - 1
    else:
        i0, i1 = _index_range_from_s_window(s_ref, last_s, s_back=s_back, s_fwd=s_fwd)

    s_closest, x_proj, y_proj, seg_i, dist = project_point_to_polyline(
        x_ref, y_ref, s_ref, x=float(x), y=float(y), i0=i0, i1=i1
    )
    s_target = float(np.clip(s_closest + float(Ld), s_ref[0], s_ref[-1]))
    x_t, y_t, yaw_t = target_point_from_s(s_ref, x_ref, y_ref, yaw_ref, s_target)

    # Heading error to target point
    heading_to_target = float(np.arctan2(y_t - float(y), x_t - float(x)))
    alpha = wrap_angle_pi(heading_to_target - float(yaw))

    # Pure pursuit curvature and steering (bicycle model)
    kappa = float(2.0 * np.sin(alpha) / float(Ld))
    delta = float(np.arctan(wheelbase_L * kappa))

    return {
        "s_closest": s_closest,
        "s_target": s_target,
        "x_proj": x_proj,
        "y_proj": y_proj,
        "proj_seg_i": int(seg_i),
        "proj_dist": dist,
        "x_target": x_t,
        "y_target": y_t,
        "yaw_target": yaw_t,
        "alpha": alpha,
        "kappa": kappa,
        "delta": delta,
    }


# --- Minimal example call (uses 'path' from the smoothing section above) ---
s_ref = path["s"]
x_ref = path["x"]
y_ref = path["y"]
yaw_ref = path.get("yaw", None)

# Example "current pose": take the first reference point, offset it a bit
x_cur = float(x_ref[0] + 20.0)
y_cur = float(y_ref[0] + -20.5)
yaw_cur = float(yaw_ref[0]) if yaw_ref is not None else 0.0

Ld = 8.0
wheelbase_L = a + b  # using your earlier parameters
out = pure_pursuit_step(
    s_ref, x_ref, y_ref, yaw_ref,
    x=x_cur, y=y_cur, yaw=yaw_cur,
    Ld=Ld, wheelbase_L=wheelbase_L,
    last_s=None,
    s_back=5.0, s_fwd=30.0,
)

print("Pure pursuit output:")
for k in ["s_closest", "s_target", "proj_dist", "alpha", "kappa", "delta"]:
    print(f"  {k:>10s}: {out[k]}")

# Visual sanity plot
plt.figure(figsize=(6, 6))
plt.plot(x_ref, y_ref, label="ref")
plt.scatter([x_cur], [y_cur], c="k", label="vehicle")
plt.scatter([out["x_target"]], [out["y_target"]], c="r", label="target")
plt.scatter([out["x_proj"]], [out["y_proj"]], c="g", label="projection")
plt.axis("equal")
plt.grid(True)
plt.legend()
plt.title("Pure pursuit geometry sanity check")
plt.show()

In [ ]:
# Offline replay: compute pure-pursuit delta over a logged run and compare to measured delta
import numpy as np
import matplotlib.pyplot as plt

# ---- Select the log segment to replay ----
gt = data_map["/EGO/gtstate"]
t_all = np.asarray(gt["time"], dtype=float)
t_all = t_all - t_all[0]

if "t_mask" in globals():
    mask = np.asarray(t_mask, dtype=bool)
else:
    mask = np.ones_like(t_all, dtype=bool)

t = t_all[mask]
x_log = np.asarray(gt["pos_x"], dtype=float)[mask]
y_log = np.asarray(gt["pos_y"], dtype=float)[mask]

# Yaw from quaternion (world frame, yaw=0 along +X)
qx = np.asarray(gt["quat_x"], dtype=float)[mask]
qy = np.asarray(gt["quat_y"], dtype=float)[mask]
qz = np.asarray(gt["quat_z"], dtype=float)[mask]
qw = np.asarray(gt["quat_w"], dtype=float)[mask]
yaw_log = np.arctan2(2.0 * (qw * qz + qx * qy), 1.0 - 2.0 * (qy**2 + qz**2))
yaw_log = np.unwrap(yaw_log)

# Measured road-wheel steer angle (front wheels average) if available
delta_meas = None
if "wheelFR_angleAtan2" in gt and "wheelFL_angleAtan2" in gt:
    delta_meas = 0.5 * (np.asarray(gt["wheelFR_angleAtan2"], dtype=float)[mask] + np.asarray(gt["wheelFL_angleAtan2"], dtype=float)[mask])

steering_input = None
if "steeringInput" in gt:
    steering_input = np.asarray(gt["steeringInput"], dtype=float)[mask]

# Speed for optional lookahead scheduling
vx_log = None
if "vel_x" in gt:
    vx_log = np.asarray(gt["vel_x"], dtype=float)[mask]

# ---- Reference path (from the smoothing section above) ----
s_ref = np.asarray(path["s"], dtype=float)
x_ref = np.asarray(path["x"], dtype=float)
y_ref = np.asarray(path["y"], dtype=float)
yaw_ref = np.asarray(path.get("yaw", None), dtype=float) if path.get("yaw", None) is not None else None

def reverse_reference(s_ref, x_ref, y_ref, yaw_ref=None):
    """Reverse travel direction while keeping s increasing."""
    s_rev = s_ref[-1] - s_ref[::-1]
    x_rev = x_ref[::-1]
    y_rev = y_ref[::-1]
    if yaw_ref is None:
        yaw_rev = None
    else:
        yaw_rev = np.unwrap(yaw_ref[::-1] + np.pi)
    return s_rev, x_rev, y_rev, yaw_rev

# Choose direction that best matches the vehicle's initial heading (avoids pi-flip confusion)
AUTO_ALIGN_DIRECTION = True
Ld_const = 8.0  # [m] lookahead for this replay (tune)
wheelbase_L = a + b

if AUTO_ALIGN_DIRECTION and yaw_ref is not None and len(t) > 0:
    x0, y0, yaw0 = float(x_log[0]), float(y_log[0]), float(yaw_log[0])
    out_fwd = pure_pursuit_step(s_ref, x_ref, y_ref, yaw_ref, x0, y0, yaw0, Ld=Ld_const, wheelbase_L=wheelbase_L)
    s_r, x_r, y_r, yaw_r = reverse_reference(s_ref, x_ref, y_ref, yaw_ref)
    out_rev = pure_pursuit_step(s_r, x_r, y_r, yaw_r, x0, y0, yaw0, Ld=Ld_const, wheelbase_L=wheelbase_L)
    if abs(out_rev["alpha"]) < abs(out_fwd["alpha"]):
        s_ref, x_ref, y_ref, yaw_ref = s_r, x_r, y_r, yaw_r
        print("Using REVERSED reference direction (better initial heading match).")
    else:
        print("Using FORWARD reference direction (better initial heading match).")

# Optional lookahead scheduling: Ld = Ld0 + k*v, clamped
USE_SPEED_SCHEDULED_LOOKAHEAD = False
Ld0 = 4.0
k_v = 0.8
Ld_min, Ld_max = 4.0, 20.0

# ---- Run pure pursuit over the log ----
delta_pp = np.zeros_like(t, dtype=float)
proj_dist = np.zeros_like(t, dtype=float)
s_closest_hist = np.zeros_like(t, dtype=float)
s_last = None

for i in range(len(t)):
    if USE_SPEED_SCHEDULED_LOOKAHEAD and vx_log is not None:
        Ld = float(np.clip(Ld0 + k_v * max(float(vx_log[i]), 0.0), Ld_min, Ld_max))
    else:
        Ld = float(Ld_const)

    out = pure_pursuit_step(
        s_ref, x_ref, y_ref, yaw_ref,
        x=float(x_log[i]), y=float(y_log[i]), yaw=float(yaw_log[i]),
        Ld=Ld, wheelbase_L=wheelbase_L,
        last_s=s_last,
        s_back=10.0, s_fwd=40.0,
    )
    delta_pp[i] = out["delta"]
    proj_dist[i] = out["proj_dist"]
    s_closest_hist[i] = out["s_closest"]
    s_last = out["s_closest"]

# ---- Plots / metrics ----
plt.figure(figsize=(12, 9))
plt.subplot(3, 1, 1)
plt.title("Steering comparison")
plt.plot(t, delta_pp, label="delta_pp (pure pursuit)")
if delta_meas is not None:
    plt.plot(t, delta_meas, label="delta_meas (front wheel)", alpha=0.8)
plt.ylabel("steer angle [rad]")
plt.grid(True)
plt.legend()

plt.subplot(3, 1, 2)
plt.plot(t, proj_dist, label="projection distance (cross-track approx)")
plt.ylabel("distance [m]")
plt.grid(True)
plt.legend()

plt.subplot(3, 1, 3)
if steering_input is not None:
    plt.plot(t, steering_input, label="steeringInput (raw)")
plt.plot(t, s_closest_hist, label="s_closest")
plt.xlabel("time [s]")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

if delta_meas is not None:
    err = delta_pp - delta_meas
    rmse = float(np.sqrt(np.mean(err**2)))
    mae = float(np.mean(np.abs(err)))
    print(f"delta RMSE: {rmse:.4f} rad, MAE: {mae:.4f} rad")

    plt.figure(figsize=(6, 6))
    plt.scatter(delta_meas, delta_pp, s=8, alpha=0.6)
    lim = float(max(np.max(np.abs(delta_meas)), np.max(np.abs(delta_pp))))
    plt.plot([-lim, lim], [-lim, lim], 'k--', linewidth=1)
    plt.xlabel("delta_meas [rad]")
    plt.ylabel("delta_pp [rad]")
    plt.title("Pure pursuit vs measured steering")
    plt.grid(True)
    plt.axis("equal")
    plt.show()

In [ ]:
from 